## v1.2 — models requiring NaN-free input (Logistic Regression, Linear SVM, Random Forest)

None of these three can accept `NaN` (sklearn's `RandomForestClassifier`
included -- being tree-based doesn't exempt it, only boosting libraries and
`HistGradientBoostingClassifier` handle `NaN` natively). So, unlike v1.1,
**every** NaN column needs a value before it reaches a model:

* `country` (6.7% missing) → explicit `"Unknown"` category, then one-hot encoded.
* `acc_balance` (8% missing) → median grouped by `country` (computed *after*
  filling `country`, so the `"Unknown"` rows form their own group too).
* `credit_score` (10.6% missing) → plain median. It has ~0 correlation with
  everything else, so there's no useful context to condition on.
* `prod_count` (5.4% missing, the highest-stakes column: churn rate is
  1 → 34.5%, 2 → 6.0%, 3 → 88.5%, 4 → 87.5%) → predicted by a
  `RandomForestClassifier` trained on rows with a known `prod_count`, using
  the other customer features (never `exit_status`). Mode-imputing this one
  would bias every unknown row toward "safe" (`prod_count`=2 has the *lowest*
  churn rate) and quietly wreck recall on the missing 5.4%.

All of this is wrapped in a single `BankChurnImputer` transformer, fit only on
training-fold data inside cross-validation, so nothing about a validation or
test row's own value ever leaks into how it gets filled.


In [1]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42


In [2]:
train = pd.read_csv('../../data/train.csv')
test = pd.read_csv('../../data/test.csv')
print("train:", train.shape, "test:", test.shape)
print(train.isna().sum())


train: (90000, 14) test: (30000, 13)
id                     0
customer_id            0
last_name              0
credit_score        9556
country             6021
gender                 0
age                    0
tenure                 0
acc_balance         7257
prod_count          4863
has_card               0
is_active              0
estimated_salary       0
exit_status            0
dtype: int64


In [3]:
ID_COLS = ['id', 'customer_id', 'last_name']
TARGET = 'exit_status'
PROD_COUNT_IMPUTE_FEATURES = [
    'age', 'is_active', 'acc_balance', 'country', 'credit_score',
    'has_card', 'estimated_salary', 'tenure',
]
NUMERIC_COLS = ['credit_score', 'age', 'tenure', 'acc_balance', 'has_card', 'is_active', 'estimated_salary']
CATEGORICAL_COLS = ['country', 'gender', 'prod_count']


def make_X(df):
    return df.drop(columns=[c for c in ID_COLS + [TARGET] if c in df.columns])


### `BankChurnImputer`

Applies the four column-specific strategies above, in dependency order
(`country` before `acc_balance`, since balance imputation groups by country;
everything before `prod_count`, since its classifier uses the other columns
as features). `prod_count` is left as a plain integer here -- the categorical
one-hot encoding happens later in the shared `ColumnTransformer`, after
imputation, so the encoder only ever sees complete categories.


In [4]:
class BankChurnImputer(BaseEstimator, TransformerMixin):
    def __init__(self, prod_count_features=PROD_COUNT_IMPUTE_FEATURES, random_state=RANDOM_STATE):
        self.prod_count_features = prod_count_features
        self.random_state = random_state

    def fit(self, X, y=None):
        X = X.copy()
        X['country'] = X['country'].fillna('Unknown')
        self.balance_median_by_country_ = X.groupby('country')['acc_balance'].median()
        self.balance_global_median_ = X['acc_balance'].median()
        self.credit_score_median_ = X['credit_score'].median()

        X['acc_balance'] = X['acc_balance'].fillna(X['country'].map(self.balance_median_by_country_))
        X['acc_balance'] = X['acc_balance'].fillna(self.balance_global_median_)
        X['credit_score'] = X['credit_score'].fillna(self.credit_score_median_)

        known = X.dropna(subset=['prod_count'])
        Xk = pd.get_dummies(known[self.prod_count_features], columns=['country'])
        self.prod_count_columns_ = Xk.columns
        yk = known['prod_count'].astype(int)

        self.prod_count_model_ = RandomForestClassifier(
            n_estimators=300, max_depth=None, min_samples_leaf=5,
            random_state=self.random_state, n_jobs=-1,
        )
        self.prod_count_model_.fit(Xk, yk)
        return self

    def transform(self, X):
        X = X.copy()
        X['country'] = X['country'].fillna('Unknown')
        X['acc_balance'] = X['acc_balance'].fillna(X['country'].map(self.balance_median_by_country_))
        X['acc_balance'] = X['acc_balance'].fillna(self.balance_global_median_)
        X['credit_score'] = X['credit_score'].fillna(self.credit_score_median_)

        missing = X['prod_count'].isna()
        if missing.any():
            Xm = pd.get_dummies(X.loc[missing, self.prod_count_features], columns=['country'])
            Xm = Xm.reindex(columns=self.prod_count_columns_, fill_value=0)
            X.loc[missing, 'prod_count'] = self.prod_count_model_.predict(Xm)
        return X


### OOF cross-validation + F1 threshold search

Same reasoning as v1.1: this competition scores F1, not ROC-AUC, so the
decision threshold is searched on out-of-fold scores instead of assumed to be
0.5. `LinearSVC` has no `predict_proba`, only `decision_function`, so the
threshold search works off whichever score each model provides.


In [5]:
def make_preprocessor():
    return ColumnTransformer([
        ('num', StandardScaler(), NUMERIC_COLS),
        ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_COLS),
    ])


def oof_threshold_search(build_model_fn, X, y, score_fn='predict_proba', n_splits=5, random_state=RANDOM_STATE):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    oof = np.zeros(len(X))

    for tr_idx, va_idx in skf.split(X, y):
        X_tr, X_va = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
        y_tr = y.iloc[tr_idx]

        imp = BankChurnImputer()
        imp.fit(X_tr)
        X_tr, X_va = imp.transform(X_tr), imp.transform(X_va)

        pre = make_preprocessor()
        X_tr_enc = pre.fit_transform(X_tr)
        X_va_enc = pre.transform(X_va)

        model = build_model_fn()
        sw = compute_sample_weight('balanced', y_tr)
        model.fit(X_tr_enc, y_tr, sample_weight=sw)

        if score_fn == 'predict_proba':
            oof[va_idx] = model.predict_proba(X_va_enc)[:, 1]
        else:
            oof[va_idx] = model.decision_function(X_va_enc)

    if score_fn == 'predict_proba':
        thresholds = np.linspace(0.02, 0.98, 97)
    else:
        thresholds = np.linspace(oof.min(), oof.max(), 97)
    f1s = [f1_score(y, oof > t) for t in thresholds]
    best = int(np.argmax(f1s))
    return oof, thresholds[best], f1s[best]


In [6]:
X = make_X(train)
y = train[TARGET]

def build_logreg():
    return LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)

_, thr_lr, f1_lr = oof_threshold_search(build_logreg, X, y, score_fn='predict_proba')
print(f"LogisticRegression: OOF F1={f1_lr:.4f}  best_threshold={thr_lr:.2f}")


LogisticRegression: OOF F1=0.6325  best_threshold=0.63


In [7]:
def build_svm():
    return LinearSVC(max_iter=5000, random_state=RANDOM_STATE)

_, thr_svm, f1_svm = oof_threshold_search(build_svm, X, y, score_fn='decision_function')
print(f"LinearSVC: OOF F1={f1_svm:.4f}  best_threshold={thr_svm:.2f}")


LinearSVC: OOF F1=0.6323  best_threshold=0.21


`SVC` (kernel SVM) is not used for the main sweep -- with ~90k training rows
its training cost scales quadratically-to-cubically and is impractical to
iterate on in a notebook. `LinearSVC` (liblinear-based) is the practical
"classic SVM" choice at this data size; a kernel `SVC` could still be tried on
a subsample if you want to compare.


In [8]:
def build_rf():
    return RandomForestClassifier(
        n_estimators=400, max_depth=None, min_samples_leaf=3,
        random_state=RANDOM_STATE, n_jobs=-1,
    )

_, thr_rf, f1_rf = oof_threshold_search(build_rf, X, y, score_fn='predict_proba')
print(f"RandomForest: OOF F1={f1_rf:.4f}  best_threshold={thr_rf:.2f}")


RandomForest: OOF F1=0.6458  best_threshold=0.48


In [9]:
results = pd.DataFrame([
    {'model': 'LogisticRegression', 'oof_f1': f1_lr, 'threshold': thr_lr, 'score_fn': 'predict_proba'},
    {'model': 'LinearSVC', 'oof_f1': f1_svm, 'threshold': thr_svm, 'score_fn': 'decision_function'},
    {'model': 'RandomForest', 'oof_f1': f1_rf, 'threshold': thr_rf, 'score_fn': 'predict_proba'},
]).sort_values('oof_f1', ascending=False).reset_index(drop=True)
results


,model,oof_f1,threshold,score_fn
0,RandomForest,0.645755,0.480000,predict_proba
1,LogisticRegression,0.632469,0.630000,predict_proba
2,LinearSVC,0.632334,0.214437,decision_function


### Final fit + submission

Refits whichever model scored best above on the full training set and writes
a submission.


In [ ]:
import os

BUILDERS = {'LogisticRegression': build_logreg, 'LinearSVC': build_svm, 'RandomForest': build_rf}
best_row = results.iloc[0]
best_name = best_row['model']
best_threshold = best_row['threshold']
best_score_fn = best_row['score_fn']
print(f"Refitting best model: {best_name} (OOF F1={best_row['oof_f1']:.4f})")

X_train_final = make_X(train)
y_train_final = train[TARGET]
X_test_final = make_X(test)

final_imputer = BankChurnImputer()
final_imputer.fit(X_train_final)
X_train_final = final_imputer.transform(X_train_final)
X_test_final = final_imputer.transform(X_test_final)

final_pre = make_preprocessor()
X_train_enc = final_pre.fit_transform(X_train_final)
X_test_enc = final_pre.transform(X_test_final)

final_model = BUILDERS[best_name]()
sw_final = compute_sample_weight('balanced', y_train_final)
final_model.fit(X_train_enc, y_train_final, sample_weight=sw_final)

if best_score_fn == 'predict_proba':
    test_scores = final_model.predict_proba(X_test_enc)[:, 1]
else:
    test_scores = final_model.decision_function(X_test_enc)
test_pred = (test_scores > best_threshold).astype(int)

os.makedirs('outputs', exist_ok=True)
submission = pd.DataFrame({'id': test['id'], 'exit_status': test_pred})
submission.to_csv(f'outputs/v1_2_{best_name.lower()}_submission.csv', index=False)
submission.head()


Refitting best model: RandomForest (OOF F1=0.6458)


,id,exit_status
0,0,0
1,1,1
2,2,1
3,3,0
4,4,0


: 